# 00_1 — Training Data Preparation

Builds the sentence corpus used by attacker models in notebooks 02 and beyond.

Each entry from the [ConvAI2](https://parl.ai/projects/personachat/) dataset is paired with a random name and intro template to produce a persona sentence. Two augmentation passes follow:
1. **Relation balance** — bring rare relations up to `TARGET_RELATION_COUNT` examples each
2. **Name balance** — ensure every name has at least `MIN_NAME_EXAMPLES` examples

**Input**  : `data/PAED/data/ConvAI2/u2t_map_all.json`, `data/names_data_&_ai.csv`  
**Output** : `data/sentences_train_text_db.parquet` + `data/sentences_train_vector_db.index`

In [ ]:
from pathlib import Path

Path("data/sentences_train_text_db.parquet").unlink(missing_ok=True)
Path("data/sentences_train_vector_db.index").unlink(missing_ok=True)

In [ ]:
import json
import re
import random
import faiss
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer

SOURCE_FILE            = 'data/PAED/data/ConvAI2/u2t_map_all.json'
NAMES_FILE             = 'data/names_data_&_ai.csv'
OUTPUT_FILE            = 'data/sentences_train_text_db.parquet'
INDEX_FILE             = 'data/sentences_train_vector_db.index'
MODEL_NAME             = 'all-mpnet-base-v2'
TARGET_RELATION_COUNT  = 500
MIN_NAME_EXAMPLES      = 200
RANDOM_SEED            = 42

In [ ]:
INTRO_TEMPLATES = [
    'I am {name}',
    'My name is {name}',
    "I'm {name}",
    '{name} here',
    'This is {name}',
]

def clean_sentence(sentence):
    text = re.sub(r' ([.,!?;:])', r'\1', sentence)
    text = re.sub(r'\bi\b', 'I', text)
    return text

Load the raw ConvAI2 dataset and the list of candidate names.

In [ ]:
raw   = json.loads(Path(SOURCE_FILE).read_text())
names = pd.read_csv(NAMES_FILE)['Prénom'].dropna().unique().tolist()

random.seed(RANDOM_SEED)

For each ConvAI2 triplet, extract the relation and object, filter malformed entries, then build a sentence by prepending a random name and intro template.

In [ ]:
records = []
for row in raw:
    t            = row['triplets'][0]
    tokens       = t['tokens']
    raw_sentence = ' '.join(tokens)
    relation     = t['label']
    obj          = ' '.join(tokens[i] for i in t['tail']).strip().lower()

    if not relation or not obj:
        continue
    if len(obj) < 2 or len(obj) > 40:
        continue
    if len(obj.split()) > 8:
        continue

    name  = random.choice(names)
    intro = random.choice(INTRO_TEMPLATES).format(name=name)
    text  = intro + '. ' + clean_sentence(raw_sentence)

    records.append({
        'text':         text,
        'name':         name,
        'relation':     relation,
        'object':       obj,
        'raw_sentence': raw_sentence,
    })

Collect into a dataframe, drop duplicate sentences, and print basic statistics.

In [ ]:
df = pd.DataFrame(records)
df = df.drop_duplicates(subset=['raw_sentence', 'relation', 'object']).reset_index(drop=True)
df.insert(0, 'id', range(len(df)))

print(f'Total rows: {len(df):,}')
print(f'Unique names: {df["name"].nunique()}')
print(f'Unique relations: {df["relation"].nunique()}')
print(f'Unique objects: {df["object"].nunique():,}')

Inspect the relation distribution — many categories are underrepresented.

In [ ]:
print('Relation distribution :')
print(df['relation'].value_counts().to_string())

Relation balancing: for each underrepresented relation, resample existing sentences with new random names until the count reaches `TARGET_RELATION_COUNT`.

In [ ]:
augmented = []

for relation, group in df.groupby('relation'):
    deficit = TARGET_RELATION_COUNT - len(group)
    if deficit <= 0:
        continue
    sampled = group.sample(n=deficit, replace=True, random_state=RANDOM_SEED)
    for _, row in sampled.iterrows():
        new_name  = random.choice(names)
        new_intro = random.choice(INTRO_TEMPLATES).format(name=new_name)
        augmented.append({
            'text':         new_intro + '. ' + clean_sentence(row['raw_sentence']),
            'name':         new_name,
            'relation':     row['relation'],
            'object':       row['object'],
            'raw_sentence': row['raw_sentence'],
        })

df = pd.concat([df, pd.DataFrame(augmented)], ignore_index=True)
df['id'] = range(len(df))

print(f'rows after relation balance : {len(df):,}')
print(f'relation counts — min : {df["relation"].value_counts().min()} | max : {df["relation"].value_counts().max()}')

Name balancing: ensure every name appears at least `MIN_NAME_EXAMPLES` times so the name classifier has equal coverage across all names.

In [ ]:
name_counts    = df['name'].value_counts()
name_augmented = []

for name in names:
    current = name_counts.get(name, 0)
    deficit = MIN_NAME_EXAMPLES - current
    if deficit <= 0:
        continue
    sampled = df.sample(n=deficit, replace=True, random_state=RANDOM_SEED)
    for _, row in sampled.iterrows():
        new_intro = random.choice(INTRO_TEMPLATES).format(name=name)
        name_augmented.append({
            'text':         new_intro + '. ' + clean_sentence(row['raw_sentence']),
            'name':         name,
            'relation':     row['relation'],
            'object':       row['object'],
            'raw_sentence': row['raw_sentence'],
        })

df = pd.concat([df, pd.DataFrame(name_augmented)], ignore_index=True)
df['id'] = range(len(df))

print(f'rows after name balance : {len(df):,}')
print(f'name counts — min : {df["name"].value_counts().min()} | max : {df["name"].value_counts().max()}')
print(f'unique names covered : {df["name"].nunique()} / {len(names)}')

Merge GPT-generated synthetic bio sentences. These add vocabulary and phrasing variety beyond what ConvAI2 provides.

In [ ]:
SYNTHETIC_FILES = [
    "data/GPT_synthetic_sentences/synthetic_bio_training_data_4000.json",
    "data/GPT_synthetic_sentences/synthetic_bio_training_data_10000_more.json",
    "data/GPT_synthetic_sentences/synthetic_bio_training_data_40000_more.json",
]

synthetic_records = []
for path in SYNTHETIC_FILES:
    for item in json.loads(Path(path).read_text()):
        synthetic_records.append({
            "text":         item["text"],
            "name":         item.get("name", ""),
            "relation":     item.get("relation", ""),
            "object":       item.get("object", ""),
            "raw_sentence": item["text"],
        })

df = pd.concat([df, pd.DataFrame(synthetic_records)], ignore_index=True)
df["id"] = range(len(df))

print(f"synthetic sentences added : {len(synthetic_records):,}")
print(f"total rows after merge    : {len(df):,}")
print(f"rows with empty name      : {(df['name'] == '').sum()}")
print(f"rows with empty object    : {(df['object'] == '').sum()}")
print(f"rows with empty relation  : {(df['relation'] == '').sum()}")

Load the embedding model. All notebooks in this project use the same model — `all-mpnet-base-v2`.

In [ ]:
model = SentenceTransformer(MODEL_NAME)

Encode all sentences into normalized 768-dimensional vectors. This takes a few minutes.

In [ ]:
embeddings = model.encode(
    df['text'].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

Build a FAISS inner-product index from the embeddings and save both the index and the metadata table to disk.

In [ ]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

df.to_parquet(OUTPUT_FILE, index=False)
faiss.write_index(index, INDEX_FILE)
print(f'Saved {len(df):,} rows → {OUTPUT_FILE}')
print(f'Saved FAISS index → {INDEX_FILE} ({index.ntotal} vectors, dim={index.d})')